# DQN on Pong

This notebook builds a Deep Q-Network agent for the ALE/Pong-v5 environment.
It follows the standard Atari preprocessing pipeline, defines a convolutional Q-network, uses experience replay, and trains with a target network.

## Pong environment

The [Pong](https://gymnasium.farama.org/environments/atari/pong/) environment is part of the [Atari environments](https://gymnasium.farama.org/environments/atari/). Please read that page first for general information.

You control the right paddle, you compete against the left paddle controlled by the computer. You each try to keep deflecting the ball away from your goal and into your opponent’s goal.

<center><img src="https://ale.farama.org/_images/pong.gif"/></center>

For a more detailed documentation, see the [AtariAge page](https://atariage.com/manual_html_page.php?SoftwareLabelID=587).

## 1. Install dependencies

Run this cell once to install the Atari-enabled [Gymnasium](https://gymnasium.farama.org/index.html) environment.

In [ ]:
!pip install "gymnasium[atari]" ale-py Pillow


## 2. Imports and device setup

Import the libraries needed for preprocessing, neural network creation, and training.

In [ ]:
import warnings
import json
warnings.filterwarnings('ignore')

import gymnasium as gym
import ale_py
import numpy as np
import collections
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

print('Using gymnasium version:', gym.__version__)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu


Check the GPU model

In [ ]:
!nvidia-smi 

## 3. Environment wrappers


Atari environments like Pong produce raw RGB screen images and run at a very high frame rate.
To make training a Deep Q-Network efficient and stable, we wrap the base environment with several preprocessing layers.

The original observations, provided by the environment, are:
- images of $210 \times 160$ in RGB colour
- Thus, we represent each observation using a numpy array of `(210, 160, 3) dtype=int8'

However, we have some problems:
1. The observations include **parts of the screen** that are not rellevant.
2. Number of states is: $256^{(210 \times 160 \times 3)}$ = $256^{100800}$! So, **reducing the image** size could help!
3. The images of the environment are in **color (RGB)**, but does color really provide any information?
4. In a single image it is not possible to know the **dynamics of the game** (i.e. direction and speed of the ball). Therefore, we must consider a sequence of several consecutive images to understand what is happening in the game.

We will use several wrappers to transform the observations. Specifically, we want to get (from the environment) observations with the following characteristics:
- Grayscale images
- Resolution $84 \times 84$
- Float images $\in [0,1]$

Wrappers:

- `MaxAndSkipEnv`: repeat the selected action for a few frames and return the max of the last frames to reduce flicker and lower effective frame rate.
- `FireResetEnv`: automatically press the game start button after reset when the Atari game requires a FIRE action to begin.
- `ProcessFrame84`: convert RGB frames to 84x84 grayscale images and crop the play area.
- `ImageToPyTorch`: reorder the image dimensions from HWC to CHW for PyTorch convolutional layers.
- `BufferWrapper`: stack the last 4 processed frames so the agent can see motion and direction.
- `ScaledFloatFrame`: normalize pixel values to the range [0, 1] for the neural network.

Each wrapper forwards most calls to the underlying environment but transforms observations, rewards, and episode semantics as needed. This makes the raw Atari game easier to use for deep reinforcement learning.


In [3]:
class FireResetEnv(gym.Wrapper):
    def __init__(self, env=None):
        super(FireResetEnv, self).__init__(env)
        assert env.unwrapped.get_action_meanings()[1] == 'FIRE'
        assert len(env.unwrapped.get_action_meanings()) >= 3

    def step(self, action):
        return self.env.step(action)

    def reset(self, *, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        obs, _, terminated, truncated, info = self.env.step(1)
        if terminated or truncated:
            obs, info = self.env.reset()
        obs, _, terminated, truncated, info = self.env.step(2)
        if terminated or truncated:
            obs, info = self.env.reset()
        return obs, info

class MaxAndSkipEnv(gym.Wrapper):
    def __init__(self, env=None, skip=4):
        super(MaxAndSkipEnv, self).__init__(env)
        self._obs_buffer = collections.deque(maxlen=2)
        self._skip = skip

    def step(self, action):
        total_reward = 0.0
        terminated = False
        truncated = False
        info = {}
        for _ in range(self._skip):
            obs, reward, term, trunc, info = self.env.step(action)
            self._obs_buffer.append(obs)
            total_reward += reward
            terminated = terminated or term
            truncated = truncated or trunc
            if terminated or truncated:
                break
        max_frame = np.max(np.stack(self._obs_buffer), axis=0)
        return max_frame, total_reward, terminated, truncated, info

    def reset(self, *, seed=None, options=None):
        self._obs_buffer.clear()
        obs, info = self.env.reset(seed=seed, options=options)
        self._obs_buffer.append(obs)
        return obs, info

class ProcessFrame84(gym.ObservationWrapper):
    def __init__(self, env=None):
        super(ProcessFrame84, self).__init__(env)
        self.observation_space = gym.spaces.Box(low=0, high=255, shape=(84, 84, 1), dtype=np.uint8)

    def observation(self, obs):
        return ProcessFrame84.process(obs)

    @staticmethod
    def process(frame):
        if frame.size == 210 * 160 * 3:
            img = np.reshape(frame, [210, 160, 3]).astype(np.float32)
        elif frame.size == 250 * 160 * 3:
            img = np.reshape(frame, [250, 160, 3]).astype(np.float32)
        else:
            raise ValueError('Unknown resolution: {}'.format(frame.size))
        img = img[:, :, 0] * 0.299 + img[:, :, 1] * 0.587 + img[:, :, 2] * 0.114
        resized_screen = cv2.resize(img, (84, 110), interpolation=cv2.INTER_AREA)
        x_t = resized_screen[18:102, :]
        x_t = np.reshape(x_t, [84, 84, 1])
        return x_t.astype(np.uint8)

class BufferWrapper(gym.ObservationWrapper):
    def __init__(self, env, n_steps, dtype=np.float32):
        super(BufferWrapper, self).__init__(env)
        self.dtype = dtype
        old_space = env.observation_space
        self.observation_space = gym.spaces.Box(old_space.low.repeat(n_steps, axis=0),
                                                old_space.high.repeat(n_steps, axis=0), dtype=dtype)

    def reset(self, *, seed=None, options=None):
        self.buffer = np.zeros_like(self.observation_space.low, dtype=self.dtype)
        obs, info = self.env.reset(seed=seed, options=options)
        return self.observation(obs), info

    def observation(self, observation):
        self.buffer[:-1] = self.buffer[1:]
        self.buffer[-1] = observation
        return self.buffer

class ImageToPyTorch(gym.ObservationWrapper):
    def __init__(self, env):
        super(ImageToPyTorch, self).__init__(env)
        old_shape = self.observation_space.shape
        self.observation_space = gym.spaces.Box(low=0, high=255, shape=(old_shape[-1], 
                                old_shape[0], old_shape[1]), dtype=np.uint8)

    def observation(self, observation):
        return np.moveaxis(observation, 2, 0)

class ScaledFloatFrame(gym.ObservationWrapper):
    def observation(self, obs):
        return np.array(obs).astype(np.float32) / 255.0

def make_env(env_name, render_mode=None):
    env = gym.make(env_name, render_mode=render_mode)
    env = MaxAndSkipEnv(env)
    env = FireResetEnv(env)
    env = ProcessFrame84(env)
    env = ImageToPyTorch(env)
    env = BufferWrapper(env, 4)
    env = ScaledFloatFrame(env)
    return env

def print_env_info(name, env):
    obs, info = env.reset()
    print(f'*** {name} Environment ***')
    print(f'Observation shape: {obs.shape}, dtype: {obs.dtype}, range: [{obs.min()}, {obs.max()}]')


## 4. Inspect the wrapped environment

Check the observation shape and value range after all wrappers are applied.

In [4]:
gym.register_envs(ale_py)
ENV_NAME = "ALE/Pong-v5"
standard_env = gym.make(ENV_NAME)
print('Standard observation shape:', standard_env.observation_space.shape)
wrapped_env = make_env(ENV_NAME)
print('Wrapped observation shape:', wrapped_env.observation_space.shape)
print_env_info('Wrapped', wrapped_env)

Standard observation shape: (210, 160, 3)
Wrapped observation shape: (4, 84, 84)
*** Wrapped Environment ***
Observation shape: (4, 84, 84), dtype: float32, range: [0.0, 0.6352941393852234]


A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


## 5. Define the DQN architecture

A convolutional neural network maps the stacked frames to Q-values for each action.

In [5]:
def make_DQN(input_shape, output_shape):
    return nn.Sequential(
        nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4),
        nn.ReLU(),
        nn.Conv2d(32, 64, kernel_size=4, stride=2),
        nn.ReLU(),
        nn.Conv2d(64, 64, kernel_size=3, stride=1),
        nn.ReLU(),
        nn.Flatten(),
        nn.Linear(64 * 7 * 7, 512),
        nn.ReLU(),
        nn.Linear(512, output_shape)
    )

test_env = make_env(ENV_NAME)
test_net = make_DQN(test_env.observation_space.shape, test_env.action_space.n)
print(test_net)

Sequential(
  (0): Conv2d(4, 32, kernel_size=(8, 8), stride=(4, 4))
  (1): ReLU()
  (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
  (3): ReLU()
  (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
  (5): ReLU()
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=3136, out_features=512, bias=True)
  (8): ReLU()
  (9): Linear(in_features=512, out_features=6, bias=True)
)


## 6. Experience replay buffer

We store transitions and sample random minibatches during training to break temporal correlations.

In [ ]:
Experience = collections.namedtuple('Experience', field_names=['state', 'action', 'reward', 'done', 'new_state'])

class ExperienceReplay:
    def __init__(self, capacity):
        self.buffer = collections.deque(maxlen=capacity)

    def __len__(self):
        return len(self.buffer)

    def append(self, experience):
        self.buffer.append(experience)

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        states, actions, rewards, dones, next_states = zip(*[self.buffer[idx] for idx in indices])
        return (np.array(states), np.array(actions), np.array(rewards, dtype=np.float32), 
                np.array(dones, dtype=np.uint8), np.array(next_states))

## 7. Agent and epsilon-greedy selection

The agent interacts with the environment, chooses actions with epsilon-greedy exploration, and stores experiences.

In [8]:
class Agent:
    def __init__(self, env, exp_replay_buffer):
        self.env = env
        self.exp_replay_buffer = exp_replay_buffer
        self._reset()

    def _reset(self):
        self.current_state, _ = self.env.reset()
        self.total_reward = 0.0

    def step(self, net, epsilon=0.0, device='cpu'):
        done_reward = None
        if np.random.random() < epsilon:
            action = self.env.action_space.sample()
        else:
            state_tensor = torch.tensor(np.array([self.current_state]), device=device)
            q_values = net(state_tensor)
            _, action_tensor = torch.max(q_values, dim=1)
            action = int(action_tensor.item())

        new_state, reward, terminated, truncated, _ = self.env.step(action)
        done = terminated or truncated
        self.total_reward += reward

        experience = Experience(self.current_state, action, reward, done, new_state)
        self.exp_replay_buffer.append(experience)
        self.current_state = new_state

        if done:
            done_reward = self.total_reward
            self._reset()

        return done_reward


## 8. Training loop

Train the Q-network using mini-batches of experiences and periodically update the target network.

In [ ]:
import wandb

# start a new wandb run to track this script
wandb.init(project="C5-RL-Pong-DQN")

In [ ]:
import datetime
print(">>> Training starts at ",datetime.datetime.now())

In [ ]:
# Hyperparameters
ENV_NAME = 'ALE/Pong-v5'
GAMMA = 0.99
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
REPLAY_SIZE = 10000
REPLAY_START_SIZE = 10000
SYNC_TARGET_FRAMES = 1000
EPS_START = 1.0
EPS_DECAY = 0.999985
EPS_MIN = 0.02
MAX_FRAMES = 500000
MEAN_REWARD_BOUND = 19.0
NUMBER_OF_REWARDS_TO_AVERAGE = 10

Main loop:

In [ ]:
# Create environment and networks
env = make_env(ENV_NAME)
net = make_DQN(env.observation_space.shape, env.action_space.n).to(device)
target_net = make_DQN(env.observation_space.shape, env.action_space.n).to(device)
target_net.load_state_dict(net.state_dict())

buffer = ExperienceReplay(REPLAY_SIZE)
agent = Agent(env, buffer)
optimizer = optim.Adam(net.parameters(), lr=LEARNING_RATE)

epsilon = EPS_START
frame_idx = 0
total_rewards = []
best_mean_reward = None

while frame_idx < MAX_FRAMES:
    frame_idx += 1
    epsilon = max(EPS_MIN, epsilon * EPS_DECAY)

    reward = agent.step(net, epsilon, device=device)
    if reward is not None:
        total_rewards.append(reward)
        mean_reward = np.mean(total_rewards[-NUMBER_OF_REWARDS_TO_AVERAGE:])
        print(f'Frame: {frame_idx} | Games: {len(total_rewards)} | Mean reward: {mean_reward:.3f} | epsilon: {epsilon:.3f}')
        wandb.log({"epsilon": epsilon, "reward_100": mean_reward, "reward": reward}, step=frame_idx)

        if best_mean_reward is None or best_mean_reward < mean_reward:
            best_mean_reward = mean_reward

        if mean_reward >= MEAN_REWARD_BOUND:
            print(f'Solved after {frame_idx} frames and {len(total_rewards)} games!')
            break

    if len(buffer) < REPLAY_START_SIZE:
        continue

    states, actions, rewards, dones, next_states = buffer.sample(BATCH_SIZE)
    states_v = torch.tensor(states, dtype=torch.float32, device=device)
    next_states_v = torch.tensor(next_states, dtype=torch.float32, device=device)
    actions_v = torch.tensor(actions, device=device)
    rewards_v = torch.tensor(rewards, device=device)
    done_mask = torch.BoolTensor(dones).to(device)

    state_action_values = net(states_v).gather(1, actions_v.unsqueeze(-1)).squeeze(-1)

    next_state_values = target_net(next_states_v).max(1)[0]
    next_state_values[done_mask] = 0.0
    next_state_values = next_state_values.detach()

    expected_state_action_values = rewards_v + GAMMA * next_state_values

    loss = nn.MSELoss()(state_action_values, expected_state_action_values)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if frame_idx % SYNC_TARGET_FRAMES == 0:
        target_net.load_state_dict(net.state_dict())

# Save model weights after training
torch.save(net.state_dict(), 'ALE_Pong_v5_dqn_solution.dat')
print('Training completed, model saved.')

Frame: 196, Games: 1, Mean reward: -21.000, epsilon: 0.997
Frame: 447, Games: 2, Mean reward: -20.500, epsilon: 0.993
Frame: 677, Games: 3, Mean reward: -20.333, epsilon: 0.990
Frame: 866, Games: 4, Mean reward: -20.500, epsilon: 0.987
Frame: 1093, Games: 5, Mean reward: -20.600, epsilon: 0.984
Frame: 1317, Games: 6, Mean reward: -20.667, epsilon: 0.980
Frame: 1513, Games: 7, Mean reward: -20.714, epsilon: 0.978
Frame: 1741, Games: 8, Mean reward: -20.625, epsilon: 0.974
Frame: 1975, Games: 9, Mean reward: -20.667, epsilon: 0.971
Frame: 2237, Games: 10, Mean reward: -20.500, epsilon: 0.967
Frame: 2528, Games: 11, Mean reward: -20.300, epsilon: 0.963
Frame: 2746, Games: 12, Mean reward: -20.400, epsilon: 0.960
Frame: 2970, Games: 13, Mean reward: -20.400, epsilon: 0.956
Frame: 3186, Games: 14, Mean reward: -20.300, epsilon: 0.953
Frame: 3421, Games: 15, Mean reward: -20.300, epsilon: 0.950
Frame: 3626, Games: 16, Mean reward: -20.300, epsilon: 0.947
Frame: 3840, Games: 17, Mean reward: 

Save the model:

In [ ]:
torch.save(net.state_dict(), ENV_NAME + ".dat")

In [ ]:
print(">>> Training ends at ",datetime.datetime.now())

In [ ]:
# Finish the wandb run, necessary in notebooks
wandb.finish()

## 9. Plot training progress

Visualize the reward curve for the last few games.

In [ ]:
plt.figure(figsize=(10, 5))
plt.title('Training reward per episode')
plt.plot(total_rewards, label='Episode reward')
if len(total_rewards) >= NUMBER_OF_REWARDS_TO_AVERAGE:
    smoothed = np.convolve(total_rewards, np.ones(NUMBER_OF_REWARDS_TO_AVERAGE) / NUMBER_OF_REWARDS_TO_AVERAGE, mode='valid')
    plt.plot(range(NUMBER_OF_REWARDS_TO_AVERAGE - 1, len(total_rewards)), smoothed, label='Smoothed reward')
plt.xlabel('Episodes')
plt.ylabel('Reward')
plt.legend()
plt.grid(True)
plt.show()

## 10. Evaluate the trained agent

Run one evaluation episode with the learned policy and print the total reward.

In [ ]:
eval_env = make_env(ENV_NAME)
net.load_state_dict(torch.load('ALE_Pong_v5_dqn_solution.dat', map_location=device))
net.eval()
state, _ = eval_env.reset()
total_reward = 0.0
terminated = False
truncated = False
while not (terminated or truncated):
    state_v = torch.tensor(np.array([state]), dtype=torch.float32, device=device)
    q_vals = net(state_v)
    action = int(torch.argmax(q_vals, dim=1).item())
    next_state, reward, terminated, truncated, _ = eval_env.step(action)
    total_reward += reward
    state = next_state

print('Evaluation total reward:', total_reward)


## 11. Test the trained agent and save GIFs

Run a few evaluation episodes with rendering enabled and save each episode as a GIF.

In [ ]:
TEST_MODEL_PATH = 'ALE_Pong_v5_dqn_solution.dat'
TEST_EPISODES = 3
TEST_OUTPUT_DIR = 'videos'
TEST_GIF_PREFIX = 'pong_dqn'
TEST_FPS = 20
TEST_MAX_STEPS = 10000


In [ ]:
def select_action(net, state, device):
    with torch.no_grad():
        state_v = torch.tensor(np.array([state]), dtype=torch.float32, device=device)
        q_vals = net(state_v)
        return int(torch.argmax(q_vals, dim=1).item())


def save_episode_gif(frames, output_path, fps=20):
    if not frames:
        raise RuntimeError("No frames were recorded. Use make_env(..., render_mode='rgb_array').")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    duration_ms = int(1000 / fps)
    images = [Image.fromarray(frame) for frame in frames]
    images[0].save(output_path, save_all=True, append_images=images[1:], duration=duration_ms, loop=0)


def run_test_episode(env, net, device, max_steps=10000):
    state, _ = env.reset()
    frames = []
    total_reward = 0.0
    terminated = False
    truncated = False
    steps = 0

    while not (terminated or truncated) and steps < max_steps:
        frame = env.render()
        if frame is not None:
            frames.append(frame)
        action = select_action(net, state, device)
        state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        steps += 1

    return total_reward, steps, frames


In [ ]:
from pathlib import Path

test_env = make_env(ENV_NAME, render_mode='rgb_array')
test_net = make_DQN(test_env.observation_space.shape, test_env.action_space.n).to(device)
test_net.load_state_dict(torch.load(TEST_MODEL_PATH, map_location=device))
test_net.eval()

for episode_idx in range(TEST_EPISODES):
    reward, steps, frames = run_test_episode(test_env, test_net, device, max_steps=TEST_MAX_STEPS)
    gif_path = Path(TEST_OUTPUT_DIR) / f'{TEST_GIF_PREFIX}_episode_{episode_idx + 1:03d}_reward_{reward:.0f}.gif'
    save_episode_gif(frames, gif_path, fps=TEST_FPS)
    print(f'Episode {episode_idx + 1}: reward={reward:.2f}, steps={steps}, saved={gif_path}')

test_env.close()


## 12. Benchmark over 100 independent episodes

Run a larger evaluation, report the average reward and standard deviation, and save GIFs for the best and worst episodes.

In [ ]:
BENCHMARK_EPISODES = 100
BENCHMARK_OUTPUT_DIR = 'videos/benchmarks'
BENCHMARK_RESULTS_PATH = 'results/notebook_benchmark_metrics.json'
BENCHMARK_GIF_PREFIX = 'pong_dqn_benchmark'


In [ ]:
benchmark_results = []
best_episode = None
worst_episode = None

benchmark_env = make_env(ENV_NAME, render_mode='rgb_array')
benchmark_net = make_DQN(benchmark_env.observation_space.shape, benchmark_env.action_space.n).to(device)
benchmark_net.load_state_dict(torch.load(TEST_MODEL_PATH, map_location=device))
benchmark_net.eval()

for episode_idx in range(BENCHMARK_EPISODES):
    reward, steps, frames = run_test_episode(benchmark_env, benchmark_net, device, max_steps=TEST_MAX_STEPS)
    result = {
        'episode': episode_idx + 1,
        'reward': float(reward),
        'steps': steps,
        'frames': len(frames),
    }
    benchmark_results.append(result)

    candidate = {**result, 'frames_data': frames}
    if best_episode is None or reward > best_episode['reward']:
        best_episode = candidate
    if worst_episode is None or reward < worst_episode['reward']:
        worst_episode = candidate

    print(f'Episode {episode_idx + 1}: reward={reward:.2f}, steps={steps}')

benchmark_env.close()

rewards = np.array([episode['reward'] for episode in benchmark_results], dtype=np.float32)
summary = {
    'episodes': BENCHMARK_EPISODES,
    'average_reward': float(np.mean(rewards)),
    'std_reward': float(np.std(rewards)),
    'min_reward': float(np.min(rewards)),
    'max_reward': float(np.max(rewards)),
}

output_dir = Path(BENCHMARK_OUTPUT_DIR)
best_gif_path = output_dir / f'{BENCHMARK_GIF_PREFIX}_best_episode_{best_episode["episode"]:03d}_reward_{best_episode["reward"]:.0f}.gif'
worst_gif_path = output_dir / f'{BENCHMARK_GIF_PREFIX}_worst_episode_{worst_episode["episode"]:03d}_reward_{worst_episode["reward"]:.0f}.gif'
save_episode_gif(best_episode['frames_data'], best_gif_path, fps=TEST_FPS)
save_episode_gif(worst_episode['frames_data'], worst_gif_path, fps=TEST_FPS)

summary['best_gif_path'] = str(best_gif_path)
summary['worst_gif_path'] = str(worst_gif_path)
benchmark_results[best_episode['episode'] - 1]['gif_path'] = str(best_gif_path)
benchmark_results[worst_episode['episode'] - 1]['gif_path'] = str(worst_gif_path)

metrics_path = Path(BENCHMARK_RESULTS_PATH)
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with metrics_path.open('w', encoding='utf-8') as f:
    json.dump({
        'experiment': 'notebook_benchmark',
        'env_name': ENV_NAME,
        'model_path': TEST_MODEL_PATH,
        'record_mode': 'best-worst',
        'summary': summary,
        'episodes': benchmark_results,
    }, f, indent=2)

print(f"Average reward: {summary['average_reward']:.2f} +/- {summary['std_reward']:.2f}")
print('Best GIF:', best_gif_path)
print('Worst GIF:', worst_gif_path)
print('Metrics saved to:', metrics_path)
